## Fake News Classifier Using LSTM



In [2]:
import pandas as pd

In [6]:
df=pd.read_csv(r"D:\CODING\PYTHON\NLP\9.LSTM\fake_or_real_news.csv")

In [7]:
df.head()

,Unnamed: 0,title,text,label
0,8476,You Can Smell Hillary’s Fear,"Daniel Greenfield, a Shillman Journalism Fello...",FAKE
1,10294,Watch The Exact Moment Paul Ryan Committed Pol...,Google Pinterest Digg Linkedin Reddit Stumbleu...,FAKE
2,3608,Kerry to go to Paris in gesture of sympathy,U.S. Secretary of State John F. Kerry said Mon...,REAL
3,10142,Bernie supporters on Twitter erupt in anger ag...,"— Kaydee King (@KaydeeKing) November 9, 2016 T...",FAKE
4,875,The Battle of New York: Why This Primary Matters,It's primary day in New York and front-runners...,REAL


In [8]:
df.shape

(6335, 4)

In [10]:
df.drop(columns=["Unnamed: 0"],inplace=True)

In [12]:
df.isnull().sum()

title    0
text     0
label    0
dtype: int64

In [13]:
###Drop Nan Values
df=df.dropna()


In [14]:
df.head()

,title,text,label
0,You Can Smell Hillary’s Fear,"Daniel Greenfield, a Shillman Journalism Fello...",FAKE
1,Watch The Exact Moment Paul Ryan Committed Pol...,Google Pinterest Digg Linkedin Reddit Stumbleu...,FAKE
2,Kerry to go to Paris in gesture of sympathy,U.S. Secretary of State John F. Kerry said Mon...,REAL
3,Bernie supporters on Twitter erupt in anger ag...,"— Kaydee King (@KaydeeKing) November 9, 2016 T...",FAKE
4,The Battle of New York: Why This Primary Matters,It's primary day in New York and front-runners...,REAL


In [15]:
from sklearn.preprocessing import LabelEncoder

la=LabelEncoder()
df["label"]=la.fit_transform(df["label"])

In [16]:
## Get the Independent Features

X=df.drop('label',axis=1)

In [17]:
## Get the Dependent features
y=df['label']

In [18]:
X.shape

(6335, 2)

In [19]:
y.shape

(6335,)

In [20]:
import tensorflow as tf

In [21]:
tf.__version__

'2.20.0'

In [22]:
from tensorflow.keras.layers import Embedding
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.preprocessing.text import one_hot
from tensorflow.keras.layers import LSTM
from tensorflow.keras.layers import Dense

In [23]:
### Vocabulary size
voc_size=5000

### Onehot Representation

In [24]:
messages=X.copy()

In [25]:
messages['title'][1]

'Watch The Exact Moment Paul Ryan Committed Political Suicide At A Trump Rally (VIDEO)'

In [26]:
messages

,title,text
0,You Can Smell Hillary’s Fear,"Daniel Greenfield, a Shillman Journalism Fello..."
1,Watch The Exact Moment Paul Ryan Committed Pol...,Google Pinterest Digg Linkedin Reddit Stumbleu...
2,Kerry to go to Paris in gesture of sympathy,U.S. Secretary of State John F. Kerry said Mon...
3,Bernie supporters on Twitter erupt in anger ag...,"— Kaydee King (@KaydeeKing) November 9, 2016 T..."
4,The Battle of New York: Why This Primary Matters,It's primary day in New York and front-runners...
...,...,...
6330,State Department says it can't find emails fro...,The State Department told the Republican Natio...
6331,The ‘P’ in PBS Should Stand for ‘Plutocratic’ ...,The ‘P’ in PBS Should Stand for ‘Plutocratic’ ...
6332,Anti-Trump Protesters Are Tools of the Oligarc...,Anti-Trump Protesters Are Tools of the Oligar...
6333,"In Ethiopia, Obama seeks progress on peace, se...","ADDIS ABABA, Ethiopia —President Obama convene..."


In [27]:
#messages.reset_index(inplace=True)

In [28]:
messages

,title,text
0,You Can Smell Hillary’s Fear,"Daniel Greenfield, a Shillman Journalism Fello..."
1,Watch The Exact Moment Paul Ryan Committed Pol...,Google Pinterest Digg Linkedin Reddit Stumbleu...
2,Kerry to go to Paris in gesture of sympathy,U.S. Secretary of State John F. Kerry said Mon...
3,Bernie supporters on Twitter erupt in anger ag...,"— Kaydee King (@KaydeeKing) November 9, 2016 T..."
4,The Battle of New York: Why This Primary Matters,It's primary day in New York and front-runners...
...,...,...
6330,State Department says it can't find emails fro...,The State Department told the Republican Natio...
6331,The ‘P’ in PBS Should Stand for ‘Plutocratic’ ...,The ‘P’ in PBS Should Stand for ‘Plutocratic’ ...
6332,Anti-Trump Protesters Are Tools of the Oligarc...,Anti-Trump Protesters Are Tools of the Oligar...
6333,"In Ethiopia, Obama seeks progress on peace, se...","ADDIS ABABA, Ethiopia —President Obama convene..."


In [29]:
import nltk
import re
from nltk.corpus import stopwords

In [30]:
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\sinha\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [31]:
### Dataset Preprocessing
from nltk.stem.porter import PorterStemmer ##stemming purpose
ps = PorterStemmer()
corpus = []
for i in range(0, len(messages)):
    review = re.sub('[^a-zA-Z]', ' ', messages['title'][i])
    review = review.lower()
    review = review.split()

    review = [ps.stem(word) for word in review if not word in stopwords.words('english')]
    review = ' '.join(review)
    corpus.append(review)

In [32]:
corpus

['smell hillari fear',
 'watch exact moment paul ryan commit polit suicid trump ralli video',
 'kerri go pari gestur sympathi',
 'berni support twitter erupt anger dnc tri warn',
 'battl new york primari matter',
 'tehran usa',
 'girl horrifi watch boyfriend left facetim',
 'britain schindler die',
 'fact check trump clinton command chief forum',
 'iran reportedli make new push uranium concess nuclear talk',
 'three clinton iowa glimps fire elud hillari clinton campaign',
 'donald trump shockingli weak deleg game somehow got even wors',
 'strong solar storm tech risk today news oct video',
 'way america prepar world war',
 'trump take cruz lightli',
 'women lead differ',
 'shock michel obama hillari caught glamor date rape promot',
 'hillari clinton huge troubl america notic sick thing hidden pictur liberti writer news',
 'iran bill obama like',
 'chart explain everyth need know partisanship america',
 'slipperi slope trump propos ban muslim',
 'episod sunday wire hail deplor special g

In [33]:
corpus[1]

'watch exact moment paul ryan commit polit suicid trump ralli video'

In [34]:
onehot_repr=[one_hot(words,voc_size)for words in corpus]
onehot_repr

[[3982, 3909, 1625],
 [90, 4879, 968, 2372, 1205, 2644, 2478, 3886, 4454, 3910, 3254],
 [1318, 225, 3696, 1374, 2708],
 [326, 2627, 841, 2528, 2256, 2679, 1068, 2450],
 [4242, 3654, 349, 3149, 1451],
 [4742, 3942],
 [3407, 4099, 90, 1211, 4994, 3509],
 [4764, 3714, 364],
 [1255, 3140, 4454, 4977, 3816, 1360, 4258],
 [4108, 1573, 134, 3654, 3254, 1572, 2917, 1682, 157],
 [4725, 4977, 2362, 2902, 4710, 892, 3909, 4977, 2683],
 [3087, 4454, 2741, 650, 3650, 200, 1428, 1630, 563, 443],
 [4960, 1034, 1791, 3386, 4046, 618, 2904, 1779, 3254],
 [958, 137, 2226, 1124, 120],
 [4454, 3427, 1964, 115],
 [727, 4504, 666],
 [3650, 2350, 4991, 3909, 811, 1682, 3503, 3098, 3324],
 [3909, 4977, 3215, 649, 137, 671, 1879, 4247, 1092, 136, 3696, 322, 2904],
 [4108, 2165, 4991, 688],
 [1504, 1171, 2376, 3859, 215, 1234, 137],
 [1060, 916, 4454, 2853, 4325, 2469],
 [3796, 50, 945, 787, 1838, 4732, 4355, 2758, 4996],
 [3909, 4977, 134, 2800, 3868, 2373, 467],
 [3654, 1425, 2850, 2776, 1532, 861, 2851, 246]

In [35]:
corpus[1]

'watch exact moment paul ryan commit polit suicid trump ralli video'

In [36]:
onehot_repr[1]

[90, 4879, 968, 2372, 1205, 2644, 2478, 3886, 4454, 3910, 3254]

### Embedding Representation

In [37]:
sent_length=20
embedded_docs=pad_sequences(onehot_repr,padding='post',maxlen=sent_length)
print(embedded_docs)

[[3982 3909 1625 ...    0    0    0]
 [  90 4879  968 ...    0    0    0]
 [1318  225 3696 ...    0    0    0]
 ...
 [ 145 4454 3541 ...    0    0    0]
 [ 674 4991 4088 ...    0    0    0]
 [1657 3213 2168 ...    0    0    0]]


In [38]:
embedded_docs[1]

array([  90, 4879,  968, 2372, 1205, 2644, 2478, 3886, 4454, 3910, 3254,
          0,    0,    0,    0,    0,    0,    0,    0,    0])

In [39]:
embedded_docs[0]

array([3982, 3909, 1625,    0,    0,    0,    0,    0,    0,    0,    0,
          0,    0,    0,    0,    0,    0,    0,    0,    0])

In [40]:
## Creating model
embedding_vector_features=40 ##features representation
model=Sequential()
model.add(Embedding(voc_size,embedding_vector_features,input_length=sent_length))
model.add(LSTM(100))
model.add(Dense(1,activation='sigmoid'))
model.compile(loss='binary_crossentropy',optimizer='adam',metrics=['accuracy'])
print(model.summary())

d:\CODING\PYTHON\PYTHON---3.11\Lib\site-packages\keras\src\layers\core\embedding.py:97: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

None


In [41]:
len(embedded_docs),y.shape

(6335, (6335,))

In [42]:
import numpy as np
X_final=np.array(embedded_docs)
y_final=np.array(y)

In [43]:
X_final.shape,y_final.shape

((6335, 20), (6335,))

In [44]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X_final, y_final, test_size=0.33, random_state=42)

### Model Training

In [46]:
### Finally Training
model.fit(X_train,y_train,validation_data=(X_test,y_test),epochs=30,batch_size=64)

Epoch 1/30


67/67 ━━━━━━━━━━━━━━━━━━━━ 2s 23ms/step - accuracy: 0.9835 - loss: 0.0661 - val_accuracy: 0.7422 - val_loss: 1.0542
Epoch 2/30
67/67 ━━━━━━━━━━━━━━━━━━━━ 2s 24ms/step - accuracy: 0.9854 - loss: 0.0609 - val_accuracy: 0.7465 - val_loss: 0.9323
Epoch 3/30
67/67 ━━━━━━━━━━━━━━━━━━━━ 2s 27ms/step - accuracy: 0.9840 - loss: 0.0633 - val_accuracy: 0.7461 - val_loss: 1.0211
Epoch 4/30
67/67 ━━━━━━━━━━━━━━━━━━━━ 2s 31ms/step - accuracy: 0.9830 - loss: 0.0640 - val_accuracy: 0.7418 - val_loss: 1.1222
Epoch 5/30
67/67 ━━━━━━━━━━━━━━━━━━━━ 2s 29ms/step - accuracy: 0.9861 - loss: 0.0574 - val_accuracy: 0.7418 - val_loss: 1.1099
Epoch 6/30
67/67 ━━━━━━━━━━━━━━━━━━━━ 2s 26ms/step - accuracy: 0.9875 - loss: 0.0514 - val_accuracy: 0.7461 - val_loss: 0.9006
Epoch 7/30
67/67 ━━━━━━━━━━━━━━━━━━━━ 2s 32ms/step - accuracy: 0.9877 - loss: 0.0506 - val_accuracy: 0.7456 - val_loss: 1.1670
Epoch 8/30
67/67 ━━━━━━━━━━━━━━━━━━━━ 2s 32ms/step - accuracy: 0.9882 - loss: 0.0493 - val_accuracy: 0.7346 - val_loss: 1.

### Adding Dropout

In [47]:
from tensorflow.keras.layers import Dropout
## Creating model
embedding_vector_features=40
model=Sequential()
model.add(Embedding(voc_size,embedding_vector_features,input_length=sent_length))
model.add(Dropout(0.3))
model.add(LSTM(100))
model.add(Dropout(0.3))
model.add(Dense(1,activation='sigmoid'))
model.compile(loss='binary_crossentropy',optimizer='adam',metrics=['accuracy'])

### Performance Metrics And Accuracy

In [48]:
y_pred=model.predict(X_test)

66/66 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step


In [49]:
y_pred=np.where(y_pred > 0.6, 1,0) ##AUC ROC Curve

In [50]:
from sklearn.metrics import confusion_matrix

In [51]:
confusion_matrix(y_test,y_pred)

array([[1071,    0],
       [1020,    0]], dtype=int64)

In [52]:
from sklearn.metrics import accuracy_score
accuracy_score(y_test,y_pred)

0.5121951219512195

In [53]:
from sklearn.metrics import classification_report
print(classification_report(y_test,y_pred))

              precision    recall  f1-score   support

           0       0.51      1.00      0.68      1071
           1       0.00      0.00      0.00      1020

    accuracy                           0.51      2091
   macro avg       0.26      0.50      0.34      2091
weighted avg       0.26      0.51      0.35      2091



d:\CODING\PYTHON\PYTHON---3.11\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
d:\CODING\PYTHON\PYTHON---3.11\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
d:\CODING\PYTHON\PYTHON---3.11\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
